# 🟪 Project 11 extension | CI evidence and missing logs

Select a GPU runtime in Colab and run these cells in order. This experiment trains a separate adapter for synthetic Build Release and Integration Engineering cases. A CI investigation is scored on JSON schema, exact evidence IDs, and whether its diagnosis or abstention matches the fixture.

Four held-out incidents each run twice: once with the diagnostic log and once with only the run summary. Each prompt also runs with and without the adapter. These are synthetic replay cases and the output is a diagnosis for review.

In [ ]:
%pip install unsloth

If Colab prompts you to restart the session after installation, restart it and continue with the next cell.

In [ ]:
from unsloth import FastLanguageModel
import torch

base_checkpoint = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_checkpoint,
    max_seq_length=512,
    load_in_4bit=True,
)
print("Model device:", next(model.parameters()).device)
print("Loaded in 4-bit:", model.is_loaded_in_4bit)

## 📂 Fixtures and scoring contract

Only the log establishes the cause. The summary establishes that the run failed. Training and held-out incidents use separate run names and different log text.

In [ ]:
import json

REASONS = {
    "build_failure", "test_failure", "artifact_missing", "policy_check_failed"
}
SYSTEM = {
    "role": "system",
    "content": (
        "Diagnose the failed CI run using only the evidence provided. Return one JSON "
        "object with the keys status, reason, and evidence_ids. When the diagnostic log "
        "supports a cause, set status to diagnosed, set reason to build_failure, "
        "test_failure, artifact_missing, or policy_check_failed, and cite the log ID. "
        "If the evidence only shows that the run failed and gives no supported cause, "
        "set status to insufficient_evidence, reason to null, and evidence_ids to []."
    ),
}

TRAIN_INCIDENTS = [
    ("spruce", "Compiler stopped: unresolved interface in billing module.", "build_failure"),
    ("larch", "Test suite: authorization UI snapshot assertion failed.", "test_failure"),
    ("alder", "Packaging expects widget.tar.gz; file absent from output directory.", "artifact_missing"),
    ("elm", "Policy gate rejected the run: required reviewer approval is missing.", "policy_check_failed"),
]
HELD_OUT_INCIDENTS = [
    ("cypress", "Linker exited with unresolved symbol in the renderer target.", "build_failure"),
    ("beech", "Integration runner exited after the inventory sync test timed out.", "test_failure"),
    ("fir", "Publish stage could not locate the generated installers.zip bundle.", "artifact_missing"),
    ("maple", "Release check rejected the run because required provenance was absent.", "policy_check_failed"),
]


def make_case(incident, with_log):
    run_id, log_text, reason = incident
    log_id = f"{run_id}-diagnostic"
    evidence = [{"id": f"{run_id}-summary", "text": "CI run completed with status failed."}]
    if with_log:
        evidence.append({"id": log_id, "text": log_text})
    return {
        "id": run_id + ("-with-log" if with_log else "-summary-only"),
        "with_log": with_log,
        "messages": [SYSTEM, {
            "role": "user",
            "content": f"Diagnose CI run {run_id}. Evidence: {json.dumps(evidence)}",
        }],
        "expected": {
            "status": "diagnosed" if with_log else "insufficient_evidence",
            "reason": reason if with_log else None,
            "evidence_ids": [log_id] if with_log else [],
        },
    }


def paired_cases(incidents):
    return [make_case(incident, with_log) for incident in incidents for with_log in (True, False)]


train_cases = paired_cases(TRAIN_INCIDENTS)
held_out_cases = paired_cases(HELD_OUT_INCIDENTS)
assert len(train_cases) == len(held_out_cases) == 8
assert {case["id"] for case in train_cases}.isdisjoint(case["id"] for case in held_out_cases)

## 🔍 Score JSON, evidence, diagnosis, and abstention

Schema validity requires the three expected keys and their supported types and values. A full pass requires schema validity, the expected status and reason, and the exact evidence ID list. Evidence-present and missing-log cases are also reported separately.

In [ ]:
def unique_object(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError(f"Duplicate JSON key: {key}")
        value[key] = item
    return value


def score_response(raw, expected):
    try:
        output = json.loads(raw, object_pairs_hook=unique_object)
    except ValueError:
        output = None

    valid = (
        isinstance(output, dict)
        and set(output) == set(expected)
        and output["status"] in ("diagnosed", "insufficient_evidence")
        and (output["reason"] is None or (
            isinstance(output["reason"], str) and output["reason"] in REASONS
        ))
        and isinstance(output["evidence_ids"], list)
        and all(isinstance(item, str) and item for item in output["evidence_ids"])
    )
    evidence_match = valid and output["evidence_ids"] == expected["evidence_ids"]
    status_reason_match = valid and (
        (output["status"], output["reason"]) == (expected["status"], expected["reason"])
    )
    return {
        "schema_valid": valid,
        "evidence_match": evidence_match,
        "status_reason_match": status_reason_match,
        "full_pass": evidence_match and status_reason_match,
    }

## 🧪 Capture the paired experiment

First capture the base replies. Then train the new adapter and generate the adapted replies using the same prompts and greedy decoding settings.

In [ ]:
def generate_reply(model, tokenizer, messages):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=80, do_sample=False)

    prompt_length = inputs["input_ids"].shape[-1]
    return tokenizer.decode(
        output[0, prompt_length:], skip_special_tokens=True
    ).strip()

base_replies = {
    case["id"]: generate_reply(model, tokenizer, case["messages"])
    for case in held_out_cases
}
print("Baseline replies:", len(base_replies))

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_dataset = Dataset.from_list([
    {
        "prompt": case["messages"],
        "completion": [{"role": "assistant", "content": json.dumps(case["expected"])}],
    }
    for case in train_cases
])

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        output_dir="/content/project11_bri_training",
        max_length=512,
        completion_only_loss=True,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        max_steps=30,
        learning_rate=2e-4,
        warmup_steps=5,
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=3407,
    ),
)

result = trainer.train()
adapter_dir = "/content/project11_bri_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Training loss: {result.training_loss:.4f}")
print("Adapter saved:", adapter_dir)

In [ ]:
from pathlib import Path

records = []
for case in held_out_cases:
    raw_replies = {
        "base": base_replies[case["id"]],
        "adapted": generate_reply(model, tokenizer, case["messages"]),
    }
    records.append({
        "case_id": case["id"],
        "with_log": case["with_log"],
        "expected": case["expected"],
        "replies": {
            version: {"raw": raw, "scores": score_response(raw, case["expected"])}
            for version, raw in raw_replies.items()
        },
    })


def summarize(records, version):
    scores = [row["replies"][version]["scores"] for row in records]
    return {
        "cases": len(scores),
        **{
            metric: sum(score[metric] for score in scores)
            for metric in ("schema_valid", "evidence_match", "status_reason_match", "full_pass")
        },
    }


groups = {
    "all": records,
    "with_log": [row for row in records if row["with_log"]],
    "summary_only": [row for row in records if not row["with_log"]],
}
summary = {
    group: {version: summarize(rows, version) for version in ("base", "adapted")}
    for group, rows in groups.items()
}
report = {
    "base_checkpoint": base_checkpoint,
    "gpu": torch.cuda.get_device_name(0),
    "train_cases": len(train_cases),
    "training_loss": result.training_loss,
    "summary": summary,
    "cases": records,
}
report_path = Path("/content/project11_bri_results.json")
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("BRI_EVAL")
print(json.dumps(summary, indent=2))
print("Detailed report:", report_path)

## 📦 Optional download

Download the JSON report to preserve per-case replies and scores.

In [ ]:
from google.colab import files

files.download(str(report_path))